In [1]:
import numpy as np
import pyvista as pv
from wireflux.utils.constants import mu0, pi, Kb,amu,mass_elec,elec
from wireflux.core.engine import MultiWireEngine
from wireflux.models.wires import Wire
from wireflux.models.newwires import NewWire
from wireflux.core.state import State
from wireflux.viz.jet_electrodes import get_jet_nozzles,jet_electrodes

import matplotlib.pyplot as plt

### Dimensional scales
L0 = 0.075 #m
r0 = 0.015 #m
I0 = 50000. #Amps
nden0 = 5e20 #m^-3
n = 11
massRatio = 1
Z1 = 40
Z2 = Z1*massRatio

### Derived scales
rho1 = nden0*amu*Z1 #kg/m^3
rho2 = nden0*amu*Z2 #kg/m^3
B0 = I0*mu0/(2*pi*L0) #tesla

vA1 = B0/np.sqrt(mu0*rho1)
vA2 = B0/np.sqrt(mu0*rho2)
vA0 = (vA1 + vA2)/2      # Band-aid, for now - JQM20260220

tau1 = L0/vA1 #s
tau2 = L0/vA2 #s
tau0 = (tau1 + tau2)/2      # Band-aid, for now -JQM20260220

### Total mass
loop_len = 0.386 #m
m1 = rho1*pi*r0*r0*loop_len
m2 = rho2*pi*r0*r0*loop_len

print("L0 (m)", L0)
print("B0 (T)", B0)
print("rho1 (kg/m^3)", rho1)
print("rho2 (kg/m^3)", rho2)
print("tau (s)", tau0)
print("vA (m/s)", vA0)
print("m1 (kg)", m1)
print("m2 (kg)", m2)

### Non-dimensional parameters
dt = .001
L = 1.
I = 1.
rho = 1.
Bp = 1.

r = r0/L0
dm1 = pi*r*r*(loop_len/L0)*rho1/n
dm2 = pi*r*r*(loop_len/L0)*rho2/n



Load_from_file =0
Time_to_load = 1.33000
################ Initial Conditions ################
if not Load_from_file:
    
    inner,outer = get_jet_nozzles()
    phi = np.linspace(0.,pi,n)

    ### Initialize paths
    mywires=[]
    plotter = pv.Plotter()
    #jet_electrodes(plotter=plotter)
    for i in range(0,4):
        start,end = np.array(inner[i]),np.array(outer[i])
        pvec = end-start
        st = np.zeros(3)
        st[0:2] = end
        R0 = np.linalg.norm(pvec)/2.
        y,z = R0*np.cos(phi),R0*np.sin(phi)
        mass = np.ones((n,1))*np.vstack(np.exp(-z/(0.75*L)))*2*dm1
        yaxis,zaxis = np.zeros((3,1)),np.zeros((3,1))
        yaxis[1,0],zaxis[2,0] = pvec[1]/(R0*2), 1.
        yaxis[0,0] = pvec[0]/(R0*2)

        path = ((y-y[0])*yaxis + z*zaxis).T + st
        
        newwire = NewWire(path/L0,path*0,mass,I,r=r,color='red')
        newwire.show(plotter=plotter)
        mywires.append(newwire)

    for i in range(4,8):
        start,end = np.array(inner[i]),np.array(outer[i])
        pvec = end-start
        st = np.zeros(3)
        st[0:2] = end
        R0 = np.linalg.norm(pvec)/2.
        y,z = R0*np.cos(phi),R0*np.sin(phi)
        mass = np.ones((n,1))*np.vstack(np.exp(-z/(0.75*L)))*2*dm2
        yaxis,zaxis = np.zeros((3,1)),np.zeros((3,1))
        yaxis[1,0],zaxis[2,0] = pvec[1]/(R0*2), 1.
        yaxis[0,0] = pvec[0]/(R0*2)

        path = ((y-y[0])*yaxis + z*zaxis).T + st
        
        newwire = NewWire(path/L0,path*0,mass,I,r=r,color='green')
        newwire.show(plotter=plotter)
        mywires.append(newwire)
        
    plotter.show()
    

    ################ Background solenoid ###############
    ##### Initialize path
    ##phi = np.linspace(0.,2*pi,50)
    ##mass = np.ones((len(phi),1))
    ##path = np.array([0.2*np.cos(phi)/L0,0.2*np.sin(phi)/L0,0*phi-.01]).T
    ##coil = Wire(path,path*0,mass,-1,is_fixed=True,r=.1)
    ####################################################



################ Load intial state ###############
st = State('spheromak_test',time=Time_to_load, load=Load_from_file)

if not Load_from_file:
    for w in mywires:
        st.items.append(w)
    #st.items.append(coil)
    st.save()
# st.show(velocity=1)
# plotter.show()
#st.save()
##################################################

########## Specify Boundary Conditions ###########
def BC(state):
    ### Boundary conditions
    for wire in state.items:
        if not wire.is_fixed:
            # Update current
            wire.I = np.sin(state.time * pi/2.)
            wire.r = .15

            # Smoothing
            #wire.smooth()                
            
            # Fix first and final segments
            wire.v[0:2,:]= 0.
            wire.v[-2:,:]= 0.

            # impervious lower boundary
            r0=0.05
            wire.v[2:-2,0][wire.p[2:-2,2] <= r0] = 0
            wire.v[2:-2,1][wire.p[2:-2,2] <= r0] = 0
            wire.v[2:-2,2][wire.p[2:-2,2] <= r0] = 0
            wire.p[2:-2,2][wire.p[2:-2,2] <= r0] = r0-.0001

            # remove large z velocities
            wire.v[2:-2,2][wire.v[2:-2,2] <= -r0] = 0
            wire.v[2:-2,0][wire.v[2:-2,2] > 25] = 0
            wire.v[2:-2,1][wire.v[2:-2,2] > 25] = 0
            wire.v[2:-2,2][wire.v[2:-2,2] > 25] = 0

            # mass BC
            wire.m[0,0], wire.m[-1,0] = pi*(wire.r)**3, pi*(wire.r)**3
            wire.total_mass = wire.m.sum()
##################################################

############## Run simulation engine #############
sim = MultiWireEngine(st,dt,bc=BC)
for i in range(1,1600):#(1,5500):
    new_st = sim.advance()
    if i%200 == 0:
        plotter = pv.Plotter()
        print(i,new_st.time,new_st.items[0].I,new_st.items[0].p[:,2].max())
        #new_st.save()
        plt.plot(new_st.items[0].p[:,2])
        plt.show()
        sim.state.show(velocity=False, plotter=plotter)#,forces=F)#new_st.show(velocity=True, plotter=plotter)#,forces=F)
        plotter.show()
plotter = pv.Plotter()
new_st.show()
plotter.show()
##################################################


################# Plot Results ###################
plt.figure(0)
plt.title("forces")
forces = sim.forceScheme()[0]
plt.plot(forces[:,0],forces[:,2])

plt.figure(1)
plt.title("position")
wire = sim.state.items[0]
plt.plot(wire.p[:,0],wire.p[:,2],'bo')
plt.show()

##new_st.show()
##mlab.show()
##################################################

L0 (m) 0.075
B0 (T) 0.13333333333333333
rho1 (kg/m^3) 3.3210778419999996e-05
rho2 (kg/m^3) 3.3210778419999996e-05
tau (s) 3.633849916463836e-06
vA (m/s) 20639.26736770236
m1 (kg) 9.061471952245886e-09
m2 (kg) 9.061471952245886e-09


TypeError: NewWire.__init__() got an unexpected keyword argument 'color'